# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and record sets from the dataset using `mlcroissant`. The metadata contains information about the dataset, including record set definitions, fields, and their unique `@id` values.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. Record sets are core tabular collections. Each record set consists of fields and columns. Refer to each by its `@id` for consistency.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print("Record set @id:", rs.id)
    print("  Name:", rs.name)
    print("  Description:", rs.description)
    print("  Fields:")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    print("---")

## 3. Data Extraction

Load data from each main record set into a pandas DataFrame. Use their `@id`s (not their names) as keys. This enables structured analysis while referring precisely to Croissant schema elements.

For demonstration, we extract the data for all record sets.

In [ ]:
from collections import OrderedDict

# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = OrderedDict()

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for first record set
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"Columns in record set {first_id}:\n", dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter records based on criteria, normalize numeric fields, and group by key fields. All steps reference fields by their `@id`.

Below, we demonstrate these processes on the first available record set.

In [ ]:
# Choose record set and numeric field by @id
selected_rs_id = record_set_ids[0] if record_set_ids else None

if selected_rs_id:
    df = dataframes[selected_rs_id]
    # Identify numeric fields from metadata
    rs_obj = next(rs for rs in dataset.record_sets if rs.id == selected_rs_id)
    numeric_fields = [f.id for f in rs_obj.fields if f.data_type in ['Integer', 'Float', 'Number']]
    print(f"Numeric fields in {selected_rs_id}: {numeric_fields}")

    # If none, fallback to all columns
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df.columns[0] if not df.empty else None

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field
        group_fields = [f.id for f in rs_obj.fields if f.data_type == 'Text']
        if group_fields:
            group_field_id = group_fields[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id}:")
                print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between relevant fields in the dataset. Below, we demonstrate plotting the distribution of a numeric field for the first record set using its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=8, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {selected_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If there is a categorical field, plot boxplot
    if group_fields and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} in {selected_rs_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrates how to load, overview, extract, analyze, and visualize the FAIR² dataset using the `mlcroissant` library. All entities are referenced by their unique `@id` for reproducibility and accuracy. 

Key observations:
- The dataset contains multiple record sets with clearly defined fields, supporting clinical, hormonal, and genetic analysis.
- Referencing entities by `@id` ensures traceability and schema consistency.
- Standard EDA and visualization can be performed with pandas and seaborn using record set and field `@id`s.

For advanced analysis, use the Croissant metadata to inform further processing, linking, and interoperability.